# VisionBridge - CTC Overfit Sanity Check (Colab)

Pre-retraining gate only. This notebook does not perform full training and does not modify VisionBridge source files. It synchronizes the repo, verifies the real processed dataset, and runs the existing `overfit_sanity.py` on 4 real examples for 150 steps.

PASS -> full training may proceed. FAIL -> diagnose the training/data/CTC pipeline first.

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_ROOT=Path('/content/VisionBridge')
if not (REPO_ROOT/'README.md').exists():
    subprocess.run(['git','clone','https://github.com/BharathWaj-K-R/VisionBridge.git',str(REPO_ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_ROOT),'checkout','main'],check=True)
    subprocess.run(['git','-C',str(REPO_ROOT),'pull','--ff-only'],check=True)
BACKEND_ROOT=REPO_ROOT/'backend'
if str(BACKEND_ROOT) not in sys.path: sys.path.insert(0,str(BACKEND_ROOT))
os.chdir(REPO_ROOT)
print('Repo:',REPO_ROOT)
print('HEAD:',subprocess.check_output(['git','-C',str(REPO_ROOT),'rev-parse','--short','HEAD'],text=True).strip())
print('BRANCH:',subprocess.check_output(['git','-C',str(REPO_ROOT),'branch','--show-current'],text=True).strip())
print('SYNC: PASS')

## 1. Verify real processed training data

The sanity test requires the same processed pose/face keypoints used by the training pipeline. Synthetic arrays are not accepted.

In [ ]:
DATA_DIR=REPO_ROOT/'data/processed/isltranslate'
CSV=DATA_DIR/'ISLTranslate.csv'
POSE_DIR=DATA_DIR/'pose'
FACE_DIR=DATA_DIR/'face'
print('Data directory:',DATA_DIR)
for p in (CSV,POSE_DIR,FACE_DIR): print(f'{p}: {"FOUND" if p.exists() else "MISSING"}')
if not CSV.exists(): raise FileNotFoundError(f'Missing metadata CSV: {CSV}')
if not POSE_DIR.exists() or not FACE_DIR.exists(): raise FileNotFoundError('Missing pose/face processed directories.')
pose_files=list(POSE_DIR.glob('*.npy')); face_files=list(FACE_DIR.glob('*.npy'))
print('Pose files:',len(pose_files)); print('Face files:',len(face_files))
if not pose_files or not face_files: raise RuntimeError('No processed pose/face .npy files found. Prepare the real dataset first.')
print('DATASET PREFLIGHT: PASS')

## 2. Run the existing 4-sample / 150-step real-data overfit test

This invokes `backend/app/training/overfit_sanity.py` exactly as defined by the repository contract. It uses the real tokenizer, model, CTC loss, and current padding-mask path. The script requires at least 20% CTC-loss reduction and a final blank ratio below 0.99.

In [ ]:
cmd=[sys.executable,'-m','app.training.overfit_sanity','--data-dir',str(DATA_DIR),'--samples','4','--steps','150']
env=os.environ.copy(); env['PYTHONPATH']=str(BACKEND_ROOT)
print('Running:', 'PYTHONPATH=backend python -m app.training.overfit_sanity --data-dir data/processed/isltranslate --samples 4 --steps 150')
result=subprocess.run(cmd,cwd=REPO_ROOT,env=env,text=True,capture_output=True)
print(result.stdout)
if result.stderr: print('STDERR:\n',result.stderr)
if result.returncode!=0:
    print('DECISION: FAIL - DO NOT START FULL RETRAINING.')
    raise RuntimeError('OVERFIT SANITY FAILED. Diagnose the training/data/CTC pipeline before full training.')
print('DECISION: PASS - FULL RETRAINING MAY PROCEED.')

## 3. Handoff record

Report these exact values to the next agent: OVERFIT SANITY PASS/FAIL, initial CTC loss, final CTC loss, loss reduction, final blank ratio, decoded predictions, and any error.

After PASS: run controlled full training with checkpoint/resume, then validate on multiple real videos, then continue BridgeAdapter personalization. After FAIL: do not spend GPU time on full training; diagnose first.